# RE_203 · A qui truquem?

**Optativa d'Aprenentatge automàtic · Bloc 2 — Supervisat · Dificultat 3 · durada: 1 o 2 sessions de 2 h**

Ets analista al departament de màrqueting d'un banc. La propera campanya ofereix un **dipòsit a
termini** per telèfon. Cada trucada costa temps i diners, i la majoria de clients diuen que no. La
directora vol una llista de clients a qui trucar.

**La pregunta:** un model pot triar a qui trucar millor que l'atzar? Amb quina mesura s'ha de jutjar, i on poses
el llindar per decidir si truques o no?

## Les dades

- **Font:** *Bank Marketing* (Moro, Cortez i Rita, 2014; UCI), a OpenML amb `data_id=46910`.
  Llicència CC BY 4.0.
- **Cada fila** és **un client** d'un banc portuguès contactat en una campanya telefònica
  (2008-2010): 45.211 files i 14 columnes. No hi ha cap identificador de persona.
- Aquesta versió **ja no porta** la durada de la trucada, que en la versió original revelava el
  resultat (no se sap quant durarà una trucada abans de fer-la).
- Documentació: <https://archive.ics.uci.edu/dataset/222/bank+marketing>

### Diccionari de columnes

| Columna | Què és |
| --- | --- |
| `SubscribeTermDeposit` | **objectiu**: si el client va contractar el dipòsit (`yes` / `no`) |
| `age` | edat |
| `job` | tipus de feina (`admin.`, `blue-collar`, `management`, `student`...; `unknown` si no consta) |
| `marital` | estat civil: `married`, `single`, `divorced` (inclou vidus) |
| `education` | estudis: `primary`, `secondary`, `tertiary`, `unknown` |
| `default` | si té crèdits impagats |
| `balance` | saldo mitjà anual, en euros (pot ser negatiu) |
| `housing`, `loan` | si té hipoteca i si té un préstec personal |
| `contact` | com se'l contacta: `cellular`, `telephone`, `unknown` |
| `campaign` | nombre de contactes amb aquest client durant aquesta campanya (inclou l'últim) |
| `pdays` | dies passats des de l'últim contacte d'una campanya anterior; `-1` vol dir que no se l'havia contactat |
| `previous` | nombre de contactes amb aquest client abans d'aquesta campanya |
| `poutcome` | resultat de la campanya anterior: `success`, `failure`, `other`, `unknown` |

### Com s'han recollit, i què cal tenir present

- Són les trucades reals d'un banc portuguès. Un mateix client podia rebre diverses trucades
  durant la campanya; aquí n'hi ha una fila amb el resultat final.
- `unknown` apareix a diverses columnes de text, però no vol dir el mateix a totes.

## Què has d'investigar

Prepara la recomanació per a la directora. Comença per la línia de base: què fa un model que no
truca mai, segons la precisió, la *precision* i el *recall*? Entrena una **regressió logística** i
un **bosc aleatori** amb un `ColumnTransformer` i compara'ls. Decideix amb quina mesura els
jutges, i explica per què la precisió no serveix.

La teoria ha d'aterrar al codi: escriu una funció que, a partir **només** de les quatre caselles
d'una matriu de confusió, calculi *accuracy*, *precision*, *recall* i F1, i comprova-les contra
scikit-learn. Després, el negoci: suposa que cada trucada costa **5 €** i cada client que contracta
en deixa **50 €** de marge. Per a cada llindar, quantes trucades faries i quin benefici tindries?
Quin llindar triaries? Acaba comparant la corba ROC i la corba *precision-recall*: quina explica
millor aquest problema?

**Per anar més enllà** (no és obligatori, i no hi ha una sola manera de fer-ho):

- `class_weight="balanced"` i baixar el llindar: són dues maneres de fer el mateix?
- Què vol dir `pdays = -1` per al model? Si ho tractes d'una altra manera, canvia res?
- Si canvies els costos de l'escenari, has de tornar a entrenar el model?

**Pistes per pensar** (no són la resposta):

- Si dos models empaten amb el llindar per defecte, potser el llindar per defecte no és el bo per comparar-los.
- El millor F1 i el millor benefici no tenen per què sortir al mateix llindar.

## Exemples per inspirar-te

Quaderns del curs on es fan servir les eines que et caldran. Mira com es treballa, no hi busquis
la resposta: les dades d'aquest repte són unes altres.

- [EX_02 · Breast cancer](../02_practica/EX_02_cancer.ipynb), exercicis 5-7: moure el llindar de decisió.
- [EX_00 · caixa d'eines](../02_practica/EX_00_caixa_eines.ipynb) §6 (`ColumnTransformer`) i §7 (matriu de confusió, *precision*, *recall*, F1 i `DummyClassifier`).
- [ML_03 · boscos](../01_teoria/ML_03_boscos.ipynb) (boscos) i [ML_04 · regressió logística](../01_teoria/ML_04_regressio_logistica.ipynb) (`predict_proba`).

## Lliurament

Lliura aquest quadern **executat**, sense errors, amb la teva anàlisi: cel·les de codi i, entre elles, cel·les de text on expliques què has fet i per què. Cada decisió que prenguis (què fas amb un buit, quina columna treus, quin model tries) ha de tenir la seva línia de justificació. Acaba amb una **conclusió** de tres a cinc frases, escrita per tu, que respongui l'encàrrec amb els números que has trobat.

> **Quan cridar en Xavi:** per a dubtes de **criteri**, no de sintaxi (per a la sintaxi hi ha la
> documentació i els exemples). Per exemple: «Quin model guanya? Em surt un a cada mesura.» «Puc fer servir el test per triar el llindar?»

La cel·la següent carrega les dades. A partir d'aquí el quadern és teu: afegeix-hi totes les
cel·les de codi i de text que necessitis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=46910, as_frame=True).frame
print(df.shape)
df.head()